# Knowledge Distillation — HPO Teacher → DARTS Student

Compresses the large HPO-optimized model (6.21M params, 85.07%) into the compact DARTS-derived architecture (128K params, 80.60%) via **Knowledge Distillation**.

Grid search over temperature T ∈ {1, 2, 4, 8} and α ∈ {0.3, 0.5, 0.7} (12 variants, ~2h on T4).

Teacher logits are precomputed once before the student training loop to avoid redundant forward passes.

The distillation logic lives in `training/distillation.py`. The grid search lives in `scripts/run_distillation.py`.

In [ ]:
# If you opened this notebook outside the cloned repository, clone it first.
# Change BRANCH if you want to run a different branch.
import os
from pathlib import Path

REPO_URL = "https://github.com/iwadas/GGSN-project.git"
BRANCH = "main"
REPO_DIR = Path("/content/GGSN-project")

if not Path("pyproject.toml").exists():
    os.chdir("/content")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

## Install Dependencies

In [ ]:
%pip install -q uv
!uv pip install --system -q optuna numpy pandas matplotlib pyyaml tqdm

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Mount Google Drive

Checkpoints are backed up on Drive from earlier experiments. This notebook restores them if missing locally.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT = Path("/content/drive/MyDrive/GGSN-project/checkpoints")
print(f"Drive backup: {DRIVE_CKPT}")
print(f"Exists: {DRIVE_CKPT.exists()}")

## Restore Checkpoints from Drive

Restore both checkpoints from Drive — teacher (HPO) and pre-trained student (DARTS).

The script loads the student from this checkpoint (fine-tuning) instead of training from scratch.

In [ ]:
import shutil

expected = [
    "checkpoints/hpo_best_baseline_cnn.pt",
    "checkpoints/darts_best_cnn.pt",
]

missing = [p for p in expected if not Path(p).exists()]
restored = []
failed = []

for p in missing:
    fname = Path(p).name
    src = DRIVE_CKPT / fname
    if src.exists():
        Path("checkpoints").mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, p)
        restored.append(fname)
        print(f"[RESTORED] {fname} <- Drive")
    else:
        failed.append(fname)
        print(f"[MISSING] {fname} not found locally or on Drive")

if not restored and not failed:
    print("All checkpoints present locally.")

if failed:
    print("\nMissing on Drive too. Run notebooks/run_all_experiments.ipynb first.")
else:
    print("\nReady to run distillation.")

## Run Knowledge Distillation

This grid-searches 12 variants (T × α combinations). Each variant trains for 30 epochs.
Estimated runtime on T4: ~2 hours.

In [ ]:
!python scripts/run_distillation.py

## Show Results

In [ ]:
import json
from IPython.display import Image, display

summary = json.loads(Path("knowledge_distillation/distillation_summary.json").read_text())
print("=== Best Variant ===")
best = summary["best_variant"]
print(json.dumps(best, indent=2))

print("\n=== All Variants ===")
for r in sorted(summary["grid_search_results"], key=lambda x: -x["test_accuracy"]):
    print(f"  T={r['temperature']:g}  α={r['alpha']:g}  "
          f"test_acc={r['test_accuracy']*100:.2f}%  "
          f"recovery={r['recovery_ratio']*100:.1f}%")

In [ ]:
display(Image("knowledge_distillation/distillation_comparison.png"))

## Backup Results to Drive

In [ ]:
DRIVE_KD = Path("/content/drive/MyDrive/GGSN-project/knowledge_distillation")
DRIVE_PLOTS = Path("/content/drive/MyDrive/GGSN-project/plots")
DRIVE_KD.mkdir(parents=True, exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

backup_files = [
    "knowledge_distillation/distillation_summary.json",
    "knowledge_distillation/distillation_comparison.png",
]
for f in backup_files:
    p = Path(f)
    if p.exists():
        dst = (DRIVE_KD if "knowledge_distillation" in f else DRIVE_PLOTS) / p.name
        shutil.copy2(p, dst)
        print(f"[BACKUP] {p.name} -> {dst}")